## Exercise 5: Geospatial wrangling and making maps

Skills: 
* More geospatial practice building on earlier skills
* Make a map with `geopandas`

References: 
* https://docs.calitp.org/data-infra/analytics_new_analysts/02-data-analysis-intermediate.html
* https://docs.calitp.org/data-infra/analytics_tools/python_libraries.html

In [2]:
#%%sh
#cd data-analyses/_shared_utils/
#make setup_env

In [3]:
import geopandas as gpd
import intake
import os
import pandas as pd
import shapely

os.environ["CALITP_BQ_MAX_BYTES"] = str(100_000_000_000)

from calitp_data_analysis.tables import tbls
from siuba import *

# Hint: if this doesn't import: refer to docs for correctly import
# cd into _shared_utils folder, run the make setup_env command
import shared_utils

## Research Question

What's the average number of trips per stop by operators in southern California? Show visualizations at the operator and county-level.
<br>**Geographic scope:** southern California counties
<br>**Deliverables:** chart(s) and map(s) showing metrics comparing across counties and also across operators. Make these visualizations using function(s).

### Prep data

* Use the same query, but grab a different set of operators. These are in southern California, so the map should zoom in counties ranging from LA to SD.
* *Hint*: for some counties, there are multiple operators. Make sure the average stop events per stop by counties is the weighted average.
* Use the same [shapefile for CA counties](https://gis.data.ca.gov/datasets/CALFIRE-Forestry::california-county-boundaries/explore?location=37.246136%2C-119.002032%2C6.12) as in Exercise 4.
* Join the data and only keep counties that have bus stops.
* If you cannot connect to the warehouse, use this dict to map feed_keys to names.
    ```
    feed_keys_to_names_dict = {
        "71d91d70ad6c07b1f9b0a618ffceef93": "Alhambra Schedule",
        "a7ba6f075198e9bf9152fab6c7faf0f6": "San Diego Schedule",
        "4f77ef02b983eccc0869c7540f98a7d0": "Big Blue Bus Schedule"
        "ae93a53469371fb3f9059d2097f66842": "OmniTrans Schedule",
        "180d48eb03829594478082dca5782ccd": "Culver City Schedule"
    }
    ```

In [6]:
feeds_to_names = shared_utils.gtfs_utils_v2.schedule_daily_feed_to_gtfs_dataset_name(
    selected_date = "2022-06-01",
    get_df = True
)[["feed_key", "name"]].drop_duplicates()
feeds_to_names.head()

,feed_key,name
0,5efaa2460085a481db5dfbf57ae78187,Kern Schedule
1,c50220b8622624dfa0c5c22859b14694,Humboldt Schedule
2,1b77ef49f5bc70038cbf15e4f5f98477,Compton Schedule
3,4b6b673ab50c016344c1adf09de2cc84,Banning Pass Schedule
4,7a7e9069dedca7a58e5a89aaa0a97256,Bay Area 511 Santa Rosa CityBus Schedule


In [7]:
OPERATORS = [
    "Alhambra Schedule", 
    "San Diego Schedule",
    "Big Blue Bus Schedule",
    "Culver City Schedule",
    "OmniTrans Schedule",
    "OCTA Schedule"
]

SUBSET_FEEDS = feeds_to_names[
    feeds_to_names.name.isin(OPERATORS)
].feed_key.tolist()

In [8]:
stops = (
    tbls.mart_gtfs.fct_daily_scheduled_stops()
    >> filter(_.feed_key.isin(SUBSET_FEEDS))
    >> filter(_.service_date == "2022-06-01")
    >> select(_.feed_key, 
              _.stop_id, _.pt_geom)
    >> collect()
)

/opt/conda/lib/python3.11/site-packages/sqlalchemy_bigquery/_types.py:101: SAWarning: Did not recognize type 'GEOGRAPHY' of column 'pt_geom'
  sqlalchemy.util.warn(


Check the type of `stops`. Is it a pandas df or geopandas gdf?

In [9]:
type(stops)

pandas.core.frame.DataFrame

In [10]:
# Turn stops into a gdf
geom = [shapely.wkt.loads(x) for x in stops.pt_geom]

stops = gpd.GeoDataFrame(
    stops, 
    geometry=geom, 
    crs="EPSG:4326"
).drop(columns="pt_geom")

Check the type of `stops`. Is it a pandas df or geopandas gdf?

What is the CRS and geometry column name?

In [11]:
type(stops)
print(f"CRS: {stops.crs}, geometry name: {stops.geometry.name}")

CRS: EPSG:4326, geometry name: geometry


In [19]:
stops.head()

,feed_key,stop_id,geometry
0,239e56d11510f71d7182a24c5621be8c,1092,POINT (-118.38664 34.04898)
1,239e56d11510f71d7182a24c5621be8c,1090,POINT (-118.39077 34.04875)
2,239e56d11510f71d7182a24c5621be8c,1061,POINT (-118.39544 34.05224)
3,239e56d11510f71d7182a24c5621be8c,1089,POINT (-118.39499 34.04938)
4,239e56d11510f71d7182a24c5621be8c,1017,POINT (-118.38411 34.05184)


### Bring in a new table from BigQuery

* In `mart_gtfs`, bring in the table called `fct_daily_scheduled_stops` for the subset of feeds defined above.
* Modify the snippet below to:
   * filter for the subset of operators
   * only keep columns: `feed_key`, `stop_id`, `stop_event_count`

In [23]:
stop_counts = (
    tbls.mart_gtfs.fct_daily_scheduled_stops()
    >> filter(_.service_date == "2022-06-01", _.feed_key.isin(SUBSET_FEEDS))
    >> select(_.feed_key, _.stop_id, _.stop_event_count)
    >> collect()
)
(stop_counts.feed_key + "_" + stop_counts.stop_id).duplicated().any()

/opt/conda/lib/python3.11/site-packages/sqlalchemy_bigquery/_types.py:101: SAWarning: Did not recognize type 'GEOGRAPHY' of column 'pt_geom'
  sqlalchemy.util.warn(


False

### Aggregate
* Write a function to aggregate to the operator level or county level, add new columns for desired metrics.
* Merge in CA shapefile to get a gdf.
* Add another `geometry` column, called `centroid`, and grab the county's centroid.
* Refer to [docs](https://geopandas.org/en/stable/docs/reference/api/geopandas.GeoDataFrame.set_geometry.html) to see how to pick which column to use as the `geometry` for the gdf, since technically, a gdf can handle multiple geometry columns.

In [33]:
COUNTIES_URL = "https://services1.arcgis.com/jUJYIo9tSA7EHvfZ/arcgis/rest/services/California_County_Boundaries/FeatureServer/0/query?outFields=*&where=1%3D1&f=geojson"
counties = gpd.read_file(COUNTIES_URL).dissolve(by="COUNTY_NAME").reset_index()

In [38]:
def get_county_statistics(stops: gpd.GeoDataFrame, stop_counts: pd.DataFrame, counties: gpd.GeoDataFrame, geographic_crs: str | int) -> gpd.GeoDataFrame:
    """Get the number of stops and stop events as a GeoDataFrame:
    
    params:
    stops - a GeoDataFrame representing stop locations with fields as follows
        feed_key - a unique identifier for a feed
        stop_id - an identifier that uniquely identified a stop within a feed. Does not necessarily identify a unique stop independent of feed_key
    stop_counts - a GeoDataFrame representing stop counts with fields as follows. It must have a CRS associated with its geometry:
        feed_key - a unique identifier for a feed
        stop_id - an identifier that uniquely identified a stop within a feed. Does not necessarily identify a unique stop independent of feed_key
        stop_event_count - the number of stop events over an arbitrary time period
    counties - a GeoDataFrame with fields as follows. It must have a CRS associated with its geometry:
        COUNTY_NAME - the name of the county
    
    returns:
    a GeoDataFrame with the following fields and geometry, and crs equivalent to that of counties:
        index - the name of the county
        stop_count - the number of stops within the county
        stop_event_count - the number of stop events occuring within the county in the same time period used in stop_counts
        county_geometry - a polygon or multipolygon representing a county
        centroid_geometry - a point representing the centroid of a county
    """
    # Associate the stop gdf with the stop event count df
    stops_with_counts = stops[["feed_key", "stop_id", stops.geometry.name]].merge(
        stop_counts[
            ["feed_key", "stop_id", "stop_event_count"]
        ],
        on=["feed_key", "stop_id"],
        how="left",
        validate="one_to_one"
    )
    # Associate each stop with a county
    stops_with_counties = stops_with_counts.to_crs(counties.crs).sjoin(
        counties[
            ["COUNTY_NAME", counties.geometry.name]
        ],
        how="left",
        predicate="within"
    )
    # Assert that each stop associated with exactly one county
    assert (stops_with_counts["feed_key"] == stops_with_counties["feed_key"]).all() and (stops_with_counts["stop_id"] == stops_with_counties["stop_id"]).all()
    assert stops_with_counties["COUNTY_NAME"].isna().sum() == 0
    # Aggregate results by county
    counties_aggregated = stops_with_counties.groupby("COUNTY_NAME")[["stop_id", "stop_event_count"]].agg(
        {"stop_id": "count", "stop_event_count": "sum"}
    ).rename(
        columns={"stop_id": "stop_count", "stop_event_name": "stop_event_count"}
    )
    # Get geometries for each county
    gdf_counties_aggregated = gpd.GeoDataFrame(
        counties_aggregated.merge(
            counties[["COUNTY_NAME", counties.geometry.name]].rename_geometry("county_geometry"), 
            how="left", 
            left_index=True, 
            right_on="COUNTY_NAME"
        ),
        geometry="county_geometry",
        crs=counties.crs
    )
    gdf_counties_aggregated["centroid_geometry"] = gdf_counties_aggregated.to_crs(geographic_crs).centroid.to_crs(gdf_counties_aggregated.crs)
    return gdf_counties_aggregated.set_index("COUNTY_NAME")

get_county_statistics(stops, stop_counts, counties, 2229)

,stop_count,stop_event_count,county_geometry,centroid_geometry
COUNTY_NAME,,,,
Los Angeles,1539,73507,"MULTIPOLYGON (((-118.53884 32.98000, -118.5388...",POINT (-118.22461 34.32255)
Orange,5221,178772,"POLYGON ((-118.11526 33.74164, -118.11502 33.7...",POINT (-117.76097 33.70301)
Riverside,4,147,"POLYGON ((-114.43559 34.07847, -114.43564 34.0...",POINT (-115.99394 33.74652)
San Bernardino,2257,57712,"POLYGON ((-115.41359 35.62499, -115.39705 35.6...",POINT (-116.17675 34.84184)
San Diego,4226,228968,"POLYGON ((-117.24152 33.44879, -117.24135 33.4...",POINT (-116.73473 33.03402)


### Visualizations
* Make one chart for comparing trips per stop by operators, and another chart for comparing it by counties. Use a function to do this.
* Make 1 map for comparing trips per stop by counties. Use `gdf.explore()` to do this.
* Visualizations should follow the Cal-ITP style guide: [styleguide example notebook](https://github.com/cal-itp/data-analyses/blob/main/starter_kit/style-guide-examples.ipynb)
* More on `folium` and `ipyleaflet`: https://github.com/jorisvandenbossche/geopandas-tutorial/blob/master/05-more-on-visualization.ipynb

In [ ]:
# To add styleguide
from calitp_data_analysis import styleguide
from calitp_data_analysis import calitp_color_palette as cp

Make a chart and a map of total stop events by county.

Make a chart and a map of stop events per stop by county.

Use a Markdown cell and write how you would summarize and interpret the visualizations.